In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import run_evaluation
from calculator import Calculator, cornell_potential_ansatz

pio.templates.default = "plotly_white"


In [ ]:
RUN_PATH = Path("../data/20260316/23")

# Choose exactly one data source mode: "combined" or "single".
DATA_MODE = "single"

FORCE_SCAN_CACHE_REBUILD = False
BUILD_SCAN_CACHE_IF_MISSING = True
LOAD_WORKERS = 1

T_MIN_VALUES = list(range(1, 10))
T_MAX_VALUES = [None, 20]

R0_T_MIN = 5
R0_T_MAX = None
R_MIN_VALUES = list(range(1, 6))
TARGET_FORCE = run_evaluation.DEFAULT_SOMMER_TARGET

OUTPUT_DIR = Path.cwd() / "interactive_html"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"RUN_PATH = {RUN_PATH.resolve()}")
print(f"DATA_MODE = {DATA_MODE}")
print(f"HTML output = {OUTPUT_DIR}")


In [ ]:
RUN_PATH = RUN_PATH.resolve()

if DATA_MODE not in {"combined", "single"}:
    raise ValueError("DATA_MODE must be either 'combined' or 'single'")

combine_equivalent_runs = DATA_MODE == "combined"

if combine_equivalent_runs:
    analysis_id, grouped_paths = run_evaluation._discover_equivalent_runs(str(RUN_PATH))
else:
    analysis_id = f"run_{run_evaluation.get_run_id(str(RUN_PATH))}"
    grouped_paths = [str(RUN_PATH)]

summary_cache = run_evaluation.load_cached_result(analysis_id)
if summary_cache is None:
    raise FileNotFoundError(
        f"No run_evaluation cache found for {analysis_id}. "
        "Run run_evaluation.get_or_calculate(...) once before using this notebook."
    )

plot_cache_path = run_evaluation.CALC_RESULT_BASE / f"{analysis_id}__plotly_potential_v1.json"
plot_output_dir = OUTPUT_DIR / analysis_id
plot_output_dir.mkdir(parents=True, exist_ok=True)

print(f"analysis_id = {analysis_id}")
print(f"mode run count = {len(grouped_paths)}")
print(f"summary cache = {run_evaluation.get_result_path(analysis_id)}")
print(f"scan cache = {plot_cache_path}")


In [ ]:
def _json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def _normalize_window_bound(value):
    if value is None or pd.isna(value):
        return None
    return int(value)


def _window_sort_key(window):
    t_min, t_max = window
    t_min = _normalize_window_bound(t_min)
    t_max = _normalize_window_bound(t_max)
    if t_min is None:
        raise ValueError("t_min must not be missing")
    if t_max is None:
        return (t_min, math.inf)
    return (t_min, t_max)


def _window_label(t_min, t_max):
    t_min = _normalize_window_bound(t_min)
    t_max = _normalize_window_bound(t_max)
    return f"t_min={t_min}, t_max={'None' if t_max is None else t_max}"


def _load_scan_cache(path: Path):
    if not path.exists():
        return None
    try:
        with path.open("r", encoding="utf-8") as handle:
            payload = json.load(handle)
    except (OSError, json.JSONDecodeError) as exc:
        print(f"ignoring unreadable scan cache {path}: {exc}")
        return None

    if payload.get("version") != 1:
        print(f"ignoring scan cache with unsupported version: {path}")
        return None

    cached_analysis_id = payload.get("analysis_id")
    if cached_analysis_id not in {None, analysis_id}:
        print(
            "ignoring scan cache for a different analysis id: "
            f"{cached_analysis_id} != {analysis_id}"
        )
        return None

    return payload


def _save_scan_cache(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, default=_json_default)
    print(f"saved scan cache: {path}")


def _build_wrt_scan(calc: Calculator, unique_rs, unique_ts):
    wrt_records = []
    for r_val in unique_rs:
        for t_val in unique_ts:
            try:
                w_var = calc.get_variable("W_R_T", R=int(r_val), T=int(t_val))
            except Exception:
                continue

            value = w_var.get()
            if value is None or not np.isfinite(value):
                continue

            err = w_var.err()
            wrt_records.append({
                "R": int(r_val),
                "T": int(t_val),
                "value": float(value),
                "err": float(err) if err is not None and np.isfinite(err) else None,
            })

    return wrt_records


def _build_effective_mass_scan(calc: Calculator, unique_rs, unique_ts):
    eff_mass_records = []
    t_values = sorted({int(t) for t in unique_ts})
    if len(t_values) < 2:
        return eff_mass_records

    for r_val in unique_rs:
        for t_val in t_values[:-1]:
            try:
                m_var = calc.get_variable("effective_mass", R=int(r_val), T=int(t_val))
            except Exception:
                continue

            value = m_var.get()
            if value is None or not np.isfinite(value):
                continue

            err = m_var.err()
            eff_mass_records.append({
                "R": int(r_val),
                "T": int(t_val),
                "t_mid": float(t_val) + 0.5,
                "value": float(value),
                "err": float(err) if err is not None and np.isfinite(err) else None,
            })

    return eff_mass_records


def _build_scan_cache():
    combined_w_temp, aggregation = run_evaluation._load_combined_w_temp(
        grouped_paths,
        verbose=True,
        prefix=f"[{RUN_PATH.name}]",
        load_workers=LOAD_WORKERS,
    )
    if combined_w_temp is None:
        raise RuntimeError("No combined W_temp data could be loaded for this run group")

    block_size = int(summary_cache.get("block_size", 1) or 1)
    calc = Calculator(
        combined_w_temp,
        n_bootstrap=run_evaluation.DEFAULT_N_BOOTSTRAP,
        step_size=block_size,
    )

    unique_rs = [int(r) for r in calc.get_unique_Rs()]
    unique_ts = [int(t) for t in calc.get_unique_Ts()]

    windows = []
    for t_min in T_MIN_VALUES:
        for t_max in T_MAX_VALUES:
            if t_max is not None and int(t_max) < int(t_min):
                continue
            windows.append((int(t_min), None if t_max is None else int(t_max)))
    windows = sorted(set(windows), key=_window_sort_key)

    wrt_records = _build_wrt_scan(calc, unique_rs, unique_ts)
    eff_mass_records = _build_effective_mass_scan(calc, unique_rs, unique_ts)

    v_records = []
    for t_min, t_max in windows:
        for r_val in unique_rs:
            try:
                v_var = calc.get_variable("V_R", R=int(r_val), t_min=int(t_min), t_max=t_max)
            except Exception:
                continue

            value = v_var.get()
            if value is None or not np.isfinite(value):
                continue

            err = v_var.err()
            fit_c = v_var.parameters.get("fit_C")
            v_records.append({
                "R": int(r_val),
                "t_min": int(t_min),
                "t_max": t_max,
                "value": float(value),
                "err": float(err) if err is not None and np.isfinite(err) else None,
                "fit_C": float(fit_c) if fit_c is not None and np.isfinite(fit_c) else None,
            })

    r0_records = []
    curve_records = []
    for r_min in [int(v) for v in R_MIN_VALUES]:
        try:
            r0_var = calc.get_variable(
                "r0",
                t_min=int(R0_T_MIN),
                t_max=R0_T_MAX,
                target_force=float(TARGET_FORCE),
                r_min=int(r_min),
            )
        except Exception:
            continue

        r0_value = r0_var.get()
        if r0_value is None or not np.isfinite(r0_value):
            continue

        r0_err = r0_var.err()
        params = r0_var.parameters.get("cornell_params", {}) or {}
        A = params.get("A")
        sigma = params.get("sigma")
        B = params.get("B")

        r0_records.append({
            "r_min": int(r_min),
            "r0": float(r0_value),
            "err": float(r0_err) if r0_err is not None and np.isfinite(r0_err) else None,
            "A": float(A) if A is not None and np.isfinite(A) else None,
            "sigma": float(sigma) if sigma is not None and np.isfinite(sigma) else None,
            "B": float(B) if B is not None and np.isfinite(B) else None,
            "t_min": int(R0_T_MIN),
            "t_max": R0_T_MAX,
            "target_force": float(TARGET_FORCE),
        })

        if A is None or sigma is None or B is None:
            continue

        fit_rs = [int(r) for r in unique_rs if int(r) >= int(r_min)]
        if not fit_rs:
            continue

        for r_val in fit_rs:
            try:
                v_var = calc.get_variable("V_R", R=int(r_val), t_min=int(R0_T_MIN), t_max=R0_T_MAX)
                v_value = v_var.get()
                v_err = v_var.err()
            except Exception:
                v_value = None
                v_err = None

            curve_records.append({
                "kind": "point",
                "r_min": int(r_min),
                "R": float(r_val),
                "V_fit": float(cornell_potential_ansatz(float(r_val), float(A), float(sigma), float(B))),
                "V": float(v_value) if v_value is not None and np.isfinite(v_value) else None,
                "err": float(v_err) if v_err is not None and np.isfinite(v_err) else None,
            })

        r_grid = np.linspace(min(fit_rs), max(fit_rs), 300)
        for r_value in r_grid:
            curve_records.append({
                "kind": "curve",
                "r_min": int(r_min),
                "R": float(r_value),
                "V_fit": float(cornell_potential_ansatz(float(r_value), float(A), float(sigma), float(B))),
                "V": None,
                "err": None,
            })

    return {
        "version": 1,
        "analysis_id": analysis_id,
        "path": str(RUN_PATH),
        "grouped_paths": grouped_paths,
        "aggregation": aggregation,
        "block_size": block_size,
        "t_windows": [
            {"t_min": int(t_min), "t_max": t_max}
            for t_min, t_max in windows
        ],
        "wrt_scan": wrt_records,
        "effective_mass_scan": eff_mass_records,
        "v_r_scan": v_records,
        "r0_scan": r0_records,
        "cornell_curves": curve_records,
    }


The next cell loads the dedicated scan cache if it already exists.

If the file is missing, the notebook rebuilds the scan data from the same combined `W_temp` data path used by `run_evaluation` so the visualization still works. Set `BUILD_SCAN_CACHE_IF_MISSING = True` to also persist that rebuilt scan cache back to `../data/calcResult/` for later runs.


In [ ]:
scan_cache = None if FORCE_SCAN_CACHE_REBUILD else _load_scan_cache(plot_cache_path)

if scan_cache is None:
    if FORCE_SCAN_CACHE_REBUILD:
        print(f"rebuilding scan cache: {plot_cache_path}")
    else:
        print(f"scan cache missing, rebuilding from grouped run data: {plot_cache_path}")
    scan_cache = _build_scan_cache()
    if BUILD_SCAN_CACHE_IF_MISSING:
        _save_scan_cache(plot_cache_path, scan_cache)
    else:
        print("built scan cache in memory only; set BUILD_SCAN_CACHE_IF_MISSING = True to save it")
else:
    print(f"loaded scan cache: {plot_cache_path}")

missing_scan_keys = []
if not scan_cache.get("wrt_scan"):
    missing_scan_keys.append("wrt_scan")
if not scan_cache.get("effective_mass_scan"):
    missing_scan_keys.append("effective_mass_scan")

if missing_scan_keys:
    print(f"{', '.join(missing_scan_keys)} missing from scan cache, rebuilding records from grouped run data")
    combined_w_temp, _ = run_evaluation._load_combined_w_temp(
        grouped_paths,
        verbose=True,
        prefix=f"[{RUN_PATH.name}]",
        load_workers=LOAD_WORKERS,
    )
    if combined_w_temp is None:
        raise RuntimeError("No combined W_temp data could be loaded for the missing scan records")

    block_size = int(summary_cache.get("block_size", 1) or 1)
    calc = Calculator(
        combined_w_temp,
        n_bootstrap=run_evaluation.DEFAULT_N_BOOTSTRAP,
        step_size=block_size,
    )
    unique_rs = [int(r) for r in calc.get_unique_Rs()]
    unique_ts = [int(t) for t in calc.get_unique_Ts()]
    scan_cache = dict(scan_cache)
    if "wrt_scan" in missing_scan_keys:
        scan_cache["wrt_scan"] = _build_wrt_scan(calc, unique_rs, unique_ts)
    if "effective_mass_scan" in missing_scan_keys:
        scan_cache["effective_mass_scan"] = _build_effective_mass_scan(calc, unique_rs, unique_ts)

    if BUILD_SCAN_CACHE_IF_MISSING:
        _save_scan_cache(plot_cache_path, scan_cache)


In [ ]:
def _wrt_dataframe(summary: dict, scan: dict) -> pd.DataFrame:
    wrt_scan = scan.get("wrt_scan", []) if scan is not None else []
    if wrt_scan:
        return pd.DataFrame(wrt_scan).sort_values(["R", "T"]).reset_index(drop=True)

    rows = []
    err_map = summary.get("W_R_T_err", {}) or {}
    for key, value in summary.get("W_R_T", {}).items():
        r_text, t_text = key.split(",")
        err = err_map.get(key)
        rows.append({
            "R": int(r_text),
            "T": int(t_text),
            "value": float(value),
            "err": float(err) if err is not None and np.isfinite(err) else None,
        })
    return pd.DataFrame(rows).sort_values(["R", "T"]).reset_index(drop=True)


wrt_df = _wrt_dataframe(summary_cache, scan_cache)
eff_mass_df = pd.DataFrame(scan_cache.get("effective_mass_scan", []))
v_scan_df = pd.DataFrame(scan_cache.get("v_r_scan", []))
r0_scan_df = pd.DataFrame(scan_cache.get("r0_scan", []))
cornell_df = pd.DataFrame(scan_cache.get("cornell_curves", []))

print(f"W_R_T rows: {len(wrt_df)}")
print(f"effective mass rows: {len(eff_mass_df)}")
print(f"V(R) scan rows: {len(v_scan_df)}")
print(f"r0 scan rows: {len(r0_scan_df)}")
print(f"Cornell curve rows: {len(cornell_df)}")


In [ ]:
def save_and_show(fig: go.Figure, name: str):
    html_path = plot_output_dir / name
    fig.write_html(html_path, include_plotlyjs="cdn", full_html=True)
    fig.show()
    print(f"saved html: {html_path}")
    return html_path


def make_wrt_plot(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("No W_R_T data found in the available caches")

    r_values = sorted(df["R"].unique())
    fig = go.Figure()
    for idx, r_value in enumerate(r_values):
        subset = df[df["R"] == r_value].sort_values("T")
        error_y = None
        if "err" in subset.columns and subset["err"].notna().any():
            error_y = {
                "type": "data",
                "array": subset["err"].fillna(0.0),
                "visible": True,
            }
        fig.add_trace(
            go.Scatter(
                x=subset["T"],
                y=subset["value"],
                mode="lines+markers",
                name=f"R = {r_value}",
                visible=idx == 0,
                error_y=error_y,
            )
        )

    buttons = []
    for idx, r_value in enumerate(r_values):
        visible = [False] * len(r_values)
        visible[idx] = True
        buttons.append({
            "label": f"R = {r_value}",
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"W(R,T) for R = {r_value}"},
            ],
        })

    fig.update_layout(
        title=f"W(R,T) for R = {r_values[0]}",
        xaxis_title="T",
        yaxis_title="W(R,T)",
        yaxis_type="log",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 220, "t": 70, "b": 60},
    )
    return fig


def make_effective_mass_plot(df: pd.DataFrame, v_scan: pd.DataFrame, t_min: int, t_max) -> go.Figure:
    if df.empty:
        raise ValueError("No effective mass data available")

    t_min = int(t_min)
    t_max = _normalize_window_bound(t_max)
    reference_subset = v_scan[
        (v_scan["t_min"] == t_min)
        & (v_scan["t_max"].isna() if t_max is None else v_scan["t_max"] == t_max)
    ].sort_values("R")
    reference_rows = {
        int(row["R"]): row
        for _, row in reference_subset.drop_duplicates(subset=["R"]).iterrows()
    }

    r_values = sorted(df["R"].unique())
    fig = go.Figure()
    trace_groups = {}

    for idx, r_value in enumerate(r_values):
        subset = df[df["R"] == r_value].sort_values("T")
        x_values = subset["t_mid"] if "t_mid" in subset.columns else subset["T"] + 0.5
        error_y = None
        if "err" in subset.columns and subset["err"].notna().any():
            error_y = {
                "type": "data",
                "array": subset["err"].fillna(0.0),
                "visible": True,
            }

        visible = idx == 0
        group = []
        group.append(len(fig.data))
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=subset["value"],
                mode="lines+markers",
                name=f"m_eff (R = {r_value})",
                visible=visible,
                error_y=error_y,
            )
        )

        ref_row = reference_rows.get(int(r_value))
        if ref_row is not None and pd.notna(ref_row.get("value")):
            x_min = float(np.min(x_values)) - 0.25
            x_max = float(np.max(x_values)) + 0.25
            if x_min == x_max:
                x_min -= 0.5
                x_max += 0.5
            x_band = [x_min, x_max]
            v_value = float(ref_row["value"])
            v_err = ref_row.get("err")
            if v_err is not None and np.isfinite(v_err):
                v_err = float(v_err)
                group.append(len(fig.data))
                fig.add_trace(
                    go.Scatter(
                        x=x_band,
                        y=[v_value - v_err, v_value - v_err],
                        mode="lines",
                        line={"width": 0},
                        hoverinfo="skip",
                        showlegend=False,
                        visible=visible,
                    )
                )
                group.append(len(fig.data))
                fig.add_trace(
                    go.Scatter(
                        x=x_band,
                        y=[v_value + v_err, v_value + v_err],
                        mode="lines",
                        line={"width": 0},
                        fill="tonexty",
                        fillcolor="rgba(214, 39, 40, 0.15)",
                        hoverinfo="skip",
                        showlegend=False,
                        visible=visible,
                    )
                )

            group.append(len(fig.data))
            fig.add_trace(
                go.Scatter(
                    x=x_band,
                    y=[v_value, v_value],
                    mode="lines",
                    line={"color": "crimson", "dash": "dash"},
                    name=f"V(R) fit ({_window_label(t_min, t_max)})",
                    visible=visible,
                )
            )

        trace_groups[r_value] = group

    total_traces = len(fig.data)
    buttons = []
    for r_value in r_values:
        visible = [False] * total_traces
        for trace_idx in trace_groups[r_value]:
            visible[trace_idx] = True
        buttons.append({
            "label": f"R = {r_value}",
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"Effective mass for R = {r_value} with V(R) reference for {_window_label(t_min, t_max)}"},
            ],
        })

    fig.update_layout(
        title=f"Effective mass for R = {r_values[0]} with V(R) reference for {_window_label(t_min, t_max)}",
        xaxis_title="T + 0.5",
        yaxis_title="m_eff(T)",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 280, "t": 70, "b": 60},
    )
    return fig


def make_vr_plot(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("No V(R) scan cache data available")

    window_table = (
        df[["t_min", "t_max"]]
        .assign(
            t_min=lambda frame: frame["t_min"].map(_normalize_window_bound),
            t_max=lambda frame: frame["t_max"].map(_normalize_window_bound),
        )
        .drop_duplicates()
        .assign(_sort=lambda frame: frame.apply(lambda row: _window_sort_key((row["t_min"], row["t_max"])), axis=1))
        .sort_values("_sort")
        .drop(columns="_sort")
        .reset_index(drop=True)
    )
    windows = [
        (_normalize_window_bound(t_min), _normalize_window_bound(t_max))
        for t_min, t_max in window_table.itertuples(index=False, name=None)
    ]

    fig = go.Figure()
    for idx, (t_min, t_max) in enumerate(windows):
        subset = df[(df["t_min"] == t_min) & (df["t_max"].isna() if t_max is None else df["t_max"] == t_max)].sort_values("R")
        error_y = None
        if "err" in subset.columns and subset["err"].notna().any():
            error_y = {
                "type": "data",
                "array": subset["err"].fillna(0.0),
                "visible": True,
            }
        fig.add_trace(
            go.Scatter(
                x=subset["R"],
                y=subset["value"],
                mode="lines+markers",
                name=_window_label(t_min, t_max),
                visible=idx == 0,
                error_y=error_y,
            )
        )

    buttons = []
    for idx, (t_min, t_max) in enumerate(windows):
        visible = [False] * len(windows)
        visible[idx] = True
        buttons.append({
            "label": _window_label(t_min, t_max),
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"V(R) for {_window_label(t_min, t_max)}"},
            ],
        })

    first_t_min, first_t_max = windows[0]
    fig.update_layout(
        title=f"V(R) for {_window_label(first_t_min, first_t_max)}",
        xaxis_title="R",
        yaxis_title="V(R)",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 260, "t": 70, "b": 60},
    )
    return fig


def make_r0_stability_plot(df: pd.DataFrame) -> go.Figure:
    if df.empty:
        raise ValueError("No r0 scan cache data available")

    ordered = df.sort_values("r_min")
    error_y = None
    if "err" in ordered.columns and ordered["err"].notna().any():
        error_y = {
            "type": "data",
            "array": ordered["err"].fillna(0.0),
            "visible": True,
        }

    fig = go.Figure(
        data=[
            go.Scatter(
                x=ordered["r_min"],
                y=ordered["r0"],
                mode="lines+markers",
                name="r0",
                error_y=error_y,
            )
        ]
    )
    fig.update_layout(
        title=f"r0 stability for t_min = {ordered['t_min'].iloc[0]}, t_max = {ordered['t_max'].iloc[0]}",
        xaxis_title="r_min",
        yaxis_title="r0/a",
        margin={"l": 60, "r": 40, "t": 70, "b": 60},
    )
    return fig


def make_cornell_plot(curves: pd.DataFrame, r0_scan: pd.DataFrame) -> go.Figure:
    if curves.empty or r0_scan.empty:
        raise ValueError("Cornell curve cache data is missing")

    r_min_values = sorted(r0_scan["r_min"].unique())
    fig = go.Figure()

    point_subset = curves[(curves["kind"] == "point") & curves["V"].notna()].sort_values("R")
    if not point_subset.empty:
        base_points = point_subset.drop_duplicates(subset=["R", "V", "err"]).sort_values("R")
        error_y = None
        if base_points["err"].notna().any():
            error_y = {
                "type": "data",
                "array": base_points["err"].fillna(0.0),
                "visible": True,
            }
        fig.add_trace(
            go.Scatter(
                x=base_points["R"],
                y=base_points["V"],
                mode="markers",
                name="V(R) data",
                visible=True,
                error_y=error_y,
            )
        )

    fit_trace_indices = {}
    marker_trace_indices = {}
    for idx, r_min in enumerate(r_min_values):
        curve_subset = curves[(curves["kind"] == "curve") & (curves["r_min"] == r_min)].sort_values("R")
        fit_trace_indices[r_min] = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=curve_subset["R"],
                y=curve_subset["V_fit"],
                mode="lines",
                name=f"Cornell fit (r_min={r_min})",
                visible=idx == 0,
            )
        )

        row = r0_scan[r0_scan["r_min"] == r_min].iloc[0]
        marker_y = None
        if pd.notna(row.get("A")) and pd.notna(row.get("sigma")) and pd.notna(row.get("B")):
            marker_y = cornell_potential_ansatz(float(row["r0"]), float(row["A"]), float(row["sigma"]), float(row["B"]))
        marker_trace_indices[r_min] = len(fig.data)
        fig.add_trace(
            go.Scatter(
                x=[row["r0"]],
                y=[marker_y],
                mode="markers",
                marker={"size": 10, "symbol": "diamond"},
                name=f"r0 (r_min={r_min})",
                visible=idx == 0,
            )
        )

    total_traces = len(fig.data)
    has_base_points = total_traces > 0 and fig.data[0].name == "V(R) data"

    buttons = []
    for r_min in r_min_values:
        visible = [False] * total_traces
        if has_base_points:
            visible[0] = True
        visible[fit_trace_indices[r_min]] = True
        visible[marker_trace_indices[r_min]] = True
        r0_value = r0_scan.loc[r0_scan["r_min"] == r_min, "r0"].iloc[0]
        buttons.append({
            "label": f"r_min = {r_min}",
            "method": "update",
            "args": [
                {"visible": visible},
                {"title": f"Cornell fit for r_min = {r_min} (r0/a = {r0_value:.4f})"},
            ],
        })

    first_r_min = r_min_values[0]
    first_r0 = r0_scan.loc[r0_scan["r_min"] == first_r_min, "r0"].iloc[0]
    fig.update_layout(
        title=f"Cornell fit for r_min = {first_r_min} (r0/a = {first_r0:.4f})",
        xaxis_title="R",
        yaxis_title="V(R)",
        updatemenus=[{
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.02,
            "xanchor": "left",
            "y": 1.0,
            "yanchor": "top",
        }],
        margin={"l": 60, "r": 260, "t": 70, "b": 60},
    )
    return fig


In [ ]:
wrt_fig = make_wrt_plot(wrt_df)
save_and_show(wrt_fig, "wrt_by_R.html")

eff_mass_fig = make_effective_mass_plot(eff_mass_df, v_scan_df, R0_T_MIN, R0_T_MAX)
save_and_show(eff_mass_fig, "effective_mass_by_R.html")

vr_fig = make_vr_plot(v_scan_df)
save_and_show(vr_fig, "potential_by_t_window.html")

#r0_fig = make_r0_stability_plot(r0_scan_df)
#save_and_show(r0_fig, "r0_vs_r_min.html")

cornell_fig = make_cornell_plot(cornell_df, r0_scan_df)
save_and_show(cornell_fig, "cornell_fit_by_r_min.html")
